# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkhlor006/Flyrank_internship_ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [24]:
import os

os.chdir("/content/Flyrank_internship_ML")

print("Current directory:")
print(os.getcwd())

print("\nTop-level folders:")
print(os.listdir("."))

Current directory:
/content/Flyrank_internship_ML

Top-level folders:
['scripts', 'requirements.txt', 'data', '02_your_first_readable_model.ipynb', 'GUIDE.md', 'work', '.github', 'AGENTS.md', 'outputs', 'CLAUDE.md', 'README.md', 'LICENSE', '01_first_look_and_discovery.ipynb', 'submission', '.git', '.gitignore', 'notebooks', 'DATA_USE.md', 'docs', 'SETUP.md', 'skills']


In [25]:
import os

print("Current directory:")
print(os.getcwd())

print("\nTop-level folders:")
print(os.listdir("."))

Current directory:
/content/Flyrank_internship_ML

Top-level folders:
['scripts', 'requirements.txt', 'data', '02_your_first_readable_model.ipynb', 'GUIDE.md', 'work', '.github', 'AGENTS.md', 'outputs', 'CLAUDE.md', 'README.md', 'LICENSE', '01_first_look_and_discovery.ipynb', 'submission', '.git', '.gitignore', 'notebooks', 'DATA_USE.md', 'docs', 'SETUP.md', 'skills']


In [26]:
!python scripts/01_prepare_features.py

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/Flyrank_internship_ML/data/processed/refresh_feature_vector.csv


In [27]:
import os

print(
    "Feature file exists:",
    os.path.exists("data/processed/refresh_feature_vector.csv")
)

Feature file exists: True


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



### Finding 1: Which Pages Will Grow?

The paper reports that a model trained on 96.6K pages that were clearly growing or declining achieved about 90% accuracy on unseen pages from the same brands and about 75% accuracy on brands the model had never seen before.

**Methodology question:** I would want to understand exactly how "growing" and "declining" were defined and over what future time window the labels were created. I would also check whether all features used by the model were available before that outcome period began. This would help confirm that the reported accuracy measures prediction of a future outcome rather than information that overlaps with the label.

I would also ask whether the unseen-brand evaluation uses a genuinely separated set of brands throughout model development, including feature selection and threshold choices. If so, the 75% result provides stronger evidence about performance on new brands; if not, the generalisation claim may be more optimistic than it appears.

### Finding 2: Refreshing Pages Actually Works

The paper reports that 7 of 9 strata showed statistically significant refresh lift, with the effect sizes reported separately by age and competition segment.

**Methodology question:** I would want to understand how the "refreshed" and "stale" groups were constructed and what outcome window was used after refresh. In particular, I would check how the label or outcome was defined and whether the comparison controls for differences between pages that were refreshed and pages that were not.

I would also check how the held-out evaluation was constructed and whether the same pages, brands, or information used to form the comparison groups could influence both the treatment definition and the outcome. This would help determine how far the result supports a directional refresh effect rather than only an observed association.

These are questions I would ask constructively because the answers determine how confidently the findings can be applied to new data and decisions.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Paper finding 1:")
print("Growth Prediction: 90% same-brand accuracy; 75% unseen-brand accuracy.")

print("\nPaper finding 2:")
print("Refreshing Pages Actually Works: 7 of 9 strata showed statistically significant lift.")

Paper finding 1:
Growth Prediction: 90% same-brand accuracy; 75% unseen-brand accuracy.

Paper finding 2:
Refreshing Pages Actually Works: 7 of 9 strata showed statistically significant lift.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


The row-level split achieved a Precision@50 of 0.90, while the client-grouped split achieved 0.40. The large drop shows that the row-level result may be optimistic because observations from the same clients can appear in both the training and test sets.

The client-grouped split held out 6 clients completely, with 26 clients used for training and zero client overlap between the two sets. I therefore treat the 0.40 client-grouped Precision@50 as the more honest estimate for this task.

This comparison does not prove that the row-level result is wrong; it shows that the validation design materially affects the measured performance.

In [29]:
import pandas as pd

feature_path = "data/processed/refresh_feature_vector.csv"

data = pd.read_csv(feature_path)

print("Prepared data shape:", data.shape)
print("Target counts:")
print(data["is_declining_label"].value_counts())
print("Clients:", data["client_id"].nunique())

Prepared data shape: (30000, 52)
Target counts:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Clients: 32


In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42


def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))
    top_idx = np.argsort(scores)[::-1][:k]

    return float(y_true[top_idx].sum() / k)


# ---------------------------------------------------------
# Helper: build the same feature matrix used in Week 5
# ---------------------------------------------------------

NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]


def make_features(frame):
    numeric = frame[NUMERIC_FEATURES].copy()

    numeric = (
        numeric
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    categorical = frame[CATEGORICAL_FEATURES].fillna("unknown").astype(str)

    categorical = pd.get_dummies(
        categorical,
        prefix=CATEGORICAL_FEATURES,
        dtype=float
    )

    return pd.concat(
        [
            numeric.reset_index(drop=True),
            categorical.reset_index(drop=True)
        ],
        axis=1
    )


# ---------------------------------------------------------
# 1. Row-level split
# ---------------------------------------------------------

row_train, row_test = train_test_split(
    data,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=data["is_declining_label"]
)

X_row_train = make_features(row_train)
X_row_test = make_features(row_test)

X_row_test = X_row_test.reindex(
    columns=X_row_train.columns,
    fill_value=0
)

y_row_train = row_train["is_declining_label"].astype(int)
y_row_test = row_test["is_declining_label"].astype(int)


row_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

row_model.fit(X_row_train, y_row_train)

row_scores = row_model.predict_proba(X_row_test)[:, 1]

row_precision50 = precision_at_k(
    y_row_test,
    row_scores,
    50
)


# ---------------------------------------------------------
# 2. Client-grouped split
# ---------------------------------------------------------

clients = data["client_id"].fillna("unknown").astype(str).unique()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(clients)

n_test_clients = max(
    1,
    int(round(len(shuffled_clients) * 0.20))
)

test_clients = set(
    shuffled_clients[:n_test_clients]
)

group_test_mask = (
    data["client_id"]
    .fillna("unknown")
    .astype(str)
    .isin(test_clients)
)

group_train = data.loc[~group_test_mask].copy()
group_test = data.loc[group_test_mask].copy()

X_group_train = make_features(group_train)
X_group_test = make_features(group_test)

X_group_test = X_group_test.reindex(
    columns=X_group_train.columns,
    fill_value=0
)

y_group_train = group_train["is_declining_label"].astype(int)
y_group_test = group_test["is_declining_label"].astype(int)


group_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

group_model.fit(X_group_train, y_group_train)

group_scores = group_model.predict_proba(X_group_test)[:, 1]

group_precision50 = precision_at_k(
    y_group_test,
    group_scores,
    50
)


# ---------------------------------------------------------
# Comparison
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "Split": [
        "Row-level",
        "Client-grouped"
    ],
    "Precision@50": [
        row_precision50,
        group_precision50
    ]
})

print(comparison.round(3))

print("\nRow-level training rows:", len(row_train))
print("Row-level test rows:", len(row_test))

print("\nGrouped training clients:", group_train["client_id"].nunique())
print("Grouped test clients:", group_test["client_id"].nunique())

overlap = (
    set(group_train["client_id"].astype(str))
    & set(group_test["client_id"].astype(str))
)

print("Grouped client overlap:", len(overlap))

            Split  Precision@50
0       Row-level           0.9
1  Client-grouped           0.4

Row-level training rows: 24000
Row-level test rows: 6000

Grouped training clients: 26
Grouped test clients: 6
Grouped client overlap: 0


Row-level Precision@50: 0.90
Client-grouped Precision@50: 0.40
Grouped client overlap: 0

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



I checked the final feature matrix for identifiers and outcome-derived variables that could leak information from the target into the model. `is_declining_label`, `trend_direction`, and `trend_pct` were excluded because they are directly related to the outcome. `content_id` and `client_id` were also excluded as predictive inputs because they are identifiers rather than behavioural features.

The audit found no forbidden fields in the final feature matrix, so the leakage check passed.

I also inspected real errors from the client-grouped test set. The model produced 395 false positives and 394 false negatives, showing that the model still makes mistakes even under the honest validation design. A false positive could lead to unnecessary review work, while a false negative could cause a declining page to be missed.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
FORBIDDEN_FEATURES = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
}

actual_features = set(X_group_train.columns)

leaked_features_present = sorted(
    actual_features.intersection(FORBIDDEN_FEATURES)
)

print("Forbidden features present in final matrix:")
print(leaked_features_present)

print("\nLeakage status:")

if leaked_features_present:
    print("FAIL — forbidden fields are present.")
else:
    print("PASS — no forbidden identifier or label-derived fields are present.")

Forbidden features present in final matrix:
[]

Leakage status:
PASS — no forbidden identifier or label-derived fields are present.


In [32]:
# Failure analysis using the honest client-grouped model

group_predictions = group_test.copy().reset_index(drop=True)

group_predictions["predicted_probability"] = group_scores

group_predictions["predicted_label"] = (
    group_scores >= 0.5
).astype(int)

group_predictions["error_type"] = np.select(
    [
        (group_predictions["is_declining_label"] == 0)
        & (group_predictions["predicted_label"] == 1),

        (group_predictions["is_declining_label"] == 1)
        & (group_predictions["predicted_label"] == 0)
    ],
    [
        "False positive",
        "False negative"
    ],
    default="Correct"
)

print("Error counts:")
print(group_predictions["error_type"].value_counts())

print("\nFalse-positive examples:")
display(
    group_predictions[
        group_predictions["error_type"] == "False positive"
    ][
        [
            "content_id",
            "is_declining_label",
            "predicted_probability",
            "error_type"
        ]
    ].head(3)
)

print("\nFalse-negative examples:")
display(
    group_predictions[
        group_predictions["error_type"] == "False negative"
    ][
        [
            "content_id",
            "is_declining_label",
            "predicted_probability",
            "error_type"
        ]
    ].head(3)
)

Error counts:
error_type
Correct           1536
False positive     395
False negative     394
Name: count, dtype: int64

False-positive examples:


,content_id,is_declining_label,predicted_probability,error_type
15,content_d7cbd76b788d,0,0.703886,False positive
17,content_c3e86d4031b6,0,0.827935,False positive
18,content_7dff534db3ae,0,0.549457,False positive



False-negative examples:


,content_id,is_declining_label,predicted_probability,error_type
1,content_326fa2fa449f,1,0.472859,False negative
11,content_0af426466565,1,0.421011,False negative
20,content_ea851c8c0ad2,1,0.479872,False negative


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



### Original claim

"Logistic Regression performs better than the baseline and can identify pages that should be refreshed."

### Safer rewritten claim

"On my original client-holdout evaluation, Logistic Regression achieved a measured Precision@50 of 0.76 compared with 0.44 for the Week-4 baseline. However, my validation audit showed that Precision@50 changed substantially from 0.90 under a row-level split to 0.40 under the stricter client-grouped split. I therefore treat the client-grouped result as the more cautious estimate of performance on unseen clients. The model provides directional decision-support evidence for prioritising pages for review; it does not prove that the model will generalise equally well to future clients or that refreshing a page will cause improved performance.

### Why I changed the claim

The original wording focused on a single evaluation result and implied broader performance than the evidence supports. The revised claim states exactly where the measured result came from, acknowledges the sensitivity to validation design, and separates predictive performance from a causal claim about what happens after a content refresh.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
validation_summary = pd.DataFrame({
    "validation_design": [
        "Row-level split",
        "Client-grouped split"
    ],
    "Precision@50": [
        row_precision50,
        group_precision50
    ]
})

print("Validation sensitivity:")
display(validation_summary.round(3))

Validation sensitivity:


,validation_design,Precision@50
0,Row-level split,0.9
1,Client-grouped split,0.4


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.